# MLP Classifier (scikit-learn)
### Features = CEFR Vocab (LazyA1) only 
### Proportion of words at each CEFR level (A1–C2)

In [1]:
from datasets import load_dataset
from transformers import set_seed

set_seed(42)

# Load data
ds = load_dataset("json", data_files=r"C:\Users\coope\Documents\Learner_Writing_Classification\cefr_lexical_sophistication\cefr-lexical-sophistication-main\wai_data\cefr_vocab_data\train_cefr_LazyA1.jsonl", split="train")
# convert CEFR vocab data to a vector and add as cefr_vec
cefr_keys = ["cefr_A1", "cefr_A2", "cefr_B1", "cefr_B2", "cefr_C1", "cefr_C2"]
ds = ds.map(lambda x: {"cefr_vec": [x[k] for k in cefr_keys]})
ds = ds.remove_columns(cefr_keys)

# drop A1 data from dataset (as I am comparing with Write and Improve data which does not have A1 data)
ds = ds.filter(lambda x: x["label"] != "A1")

label_map = {"A2": 0, "B1": 1, "B2": 2, "C1": 3}
ds = ds.map(lambda x: {"labels": label_map[x["label"]]})
ds = ds.select_columns(["cefr_vec", "labels"])

In [2]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np

X_train = np.array(ds["cefr_vec"])
y_train = np.array(ds["labels"])

clf = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=500, random_state=42)
)
clf.fit(X_train, y_train)

,steps,"[('standardscaler', ...), ('mlpclassifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,hidden_layer_sizes,"(256, ...)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001


In [3]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import plotly.graph_objects as go
import numpy as np

# Load test data
test_ds = load_dataset("json", data_files=r"C:\Users\coope\Documents\Learner_Writing_Classification\cefr_lexical_sophistication\cefr-lexical-sophistication-main\wai_data\cefr_vocab_data\test_cefr_LazyA1.jsonl", split="train")

# convert CEFR vocab data to a vector and add as cefr_vec
cefr_keys = ["cefr_A1", "cefr_A2", "cefr_B1", "cefr_B2", "cefr_C1", "cefr_C2"]
test_ds = test_ds.map(lambda x: {"cefr_vec": [x[k] for k in cefr_keys]})
test_ds = test_ds.remove_columns(cefr_keys)

# drop A1 data from dataset (as I am comparing with Write and Improve data which does not have A1 data)
test_ds = test_ds.filter(lambda x: x["label"] != "A1")

label_map = {"A2": 0, "B1": 1, "B2": 2, "C1": 3}
test_ds = test_ds.map(lambda x: {"labels": label_map[x["label"]]})
test_ds = test_ds.select_columns(["cefr_vec", "labels"])

# Get predictions
X_test = np.array(test_ds["cefr_vec"])
all_labels = list(test_ds["labels"])
all_preds = list(clf.predict(X_test))

# Calculate metrics
precision, recall, f1, support = precision_recall_fscore_support(all_labels, all_preds, average=None)
labels = ["A2", "B1", "B2", "C1"]

print("Per-class metrics:")
for i, label in enumerate(labels):
    print(f"{label}: Precision={precision[i]:.3f}, Recall={recall[i]:.3f}, F1={f1[i]:.3f}")

# Overall metrics
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro")
print(f"\nMacro average: Precision={precision_macro:.3f}, Recall={recall_macro:.3f}, F1={f1_macro:.3f}")

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
# Normalize by row (true labels) to get proportions
cm_normalized = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

# Create text annotations with both count and percentage
annotations = []
for i in range(len(labels)):
    row = []
    for j in range(len(labels)):
        row.append(f"{cm_normalized[i][j]*100:.1f}%<br>({cm[i][j]})")
    annotations.append(row)

# Create plotly heatmap (black = high, white = low)
fig = go.Figure(data=go.Heatmap(
    z=cm_normalized,
    x=labels,
    y=labels,
    colorscale=[[0, "white"], [1, "black"]],
    text=annotations,
    texttemplate="%{text}",
    textfont={"size": 12},
    hovertemplate="True: %{y}<br>Predicted: %{x}<br>Proportion: %{z:.2f}<extra></extra>"
))

fig.update_layout(
    title="CEFR Vocabulary Only (LazyA1)",
    xaxis_title="Predicted",
    yaxis_title="True",
    xaxis={"side": "bottom"},
    yaxis={"autorange": "reversed"},
    width=500,
    height=500
)

fig.show()

Filter:   0%|          | 0/404 [00:00<?, ? examples/s]

Map:   0%|          | 0/404 [00:00<?, ? examples/s]

Per-class metrics:
A2: Precision=0.722, Recall=0.605, F1=0.658
B1: Precision=0.484, Recall=0.392, F1=0.434
B2: Precision=0.397, Recall=0.536, F1=0.456
C1: Precision=0.340, Recall=0.375, F1=0.356

Macro average: Precision=0.486, Recall=0.477, F1=0.476


# QWK

In [4]:
# %%
from sklearn.metrics import cohen_kappa_score

qwk = cohen_kappa_score(all_labels, all_preds, weights="quadratic")
print(f"Quadratic Weighted Kappa: {qwk:.3f}")

Quadratic Weighted Kappa: 0.556


# Save Predictions

In [5]:
import pandas as pd

id_to_label = {0: "A2", 1: "B1", 2: "B2", 3: "C1"}

# reload in test_ds
test_ds = load_dataset("json", data_files=r"C:\Users\coope\Documents\Learner_Writing_Classification\cefr_lexical_sophistication\cefr-lexical-sophistication-main\wai_data\cefr_vocab_data\test_cefr_LazyA1.jsonl", split="train")


pd.DataFrame({
    "id": test_ds["id"],
    "topic": test_ds["topic"],
    "genre": test_ds["genre"],
    "true": [id_to_label[l] for l in all_labels],
    "pred": [id_to_label[p] for p in all_preds]
}).to_csv("predictions/AW_only_preds.csv", index=False)
